In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import numpy as np

c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

# Knowledge base
docs = [
    "AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.",
    "The system can use satellite data, weather data, water temperature, and sensor measurements.",
    "Dense retrieval uses embeddings to find documents with similar meaning.",
    "BM25 is a sparse retrieval method based on keyword matching.",
    "Transformers are neural network architectures used in models such as GPT, BERT, and LLaMA."
]

query = "How does AI4HAB predict algae blooms?"

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedding_model.encode(docs)
query_embedding = embedding_model.encode([query])

# Similarity search
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k = 2
top_indices = np.argsort(similarities)[::-1][:top_k]
retrieved_docs = [docs[i] for i in top_indices]

print("Retrieved Documents:\n")
for doc in retrieved_docs:
    print("-", doc)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2991.35it/s]


Retrieved Documents:

- AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.
- The system can use satellite data, weather data, water temperature, and sensor measurements.


In [ ]:

# Build context
context = " ".join(retrieved_docs)

# This is what LLM will see as input
# Improved prompt
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Write one clear sentence.
Answer:
"""

# Better text generation model
generator = pipeline(
    "text-generation",
    model="microsoft/Phi-2" #try different models
)

answer = generator(
    prompt,
    max_new_tokens=80,
    do_sample=False 
)[0]["generated_text"]

print("\nGenerated Answers:\n")
print(answer)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--microsoft--Phi-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
